# 06a. Sequence model

A small transformer that reads each transaction's recent history: the last transactions of the sender and of the receiver, alongside the current row's 22 features. Its purpose is to test whether that history carries signal the 22 features miss, in particular for laundering from hub senders, which every model so far fails to detect. The design was fixed before any results (see the PRD). This notebook first measures how much history accounts actually have, to set the sequence length.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from bankguard.data import load_transactions, split_by_date
from bankguard.features import BIG, account_codes, count_prior, minutes_from_start

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
val = split_by_date(tx)["validation"]

sender, receiver = account_codes(tx)
minutes = minutes_from_start(tx)
self_tx = tx["self_transfer"].to_numpy()

# Every transaction is one event for the sender and one for the receiver.
# A self transfer is the same account on both sides, so it counts once.
event_codes = np.concatenate([sender, receiver[~self_tx]])
event_minutes = np.concatenate([minutes, minutes[~self_tx]])

pos = val.index.to_numpy()
history = pd.DataFrame({
    "sender": count_prior(event_codes, event_minutes, sender[pos], minutes[pos], BIG),
    "receiver": count_prior(event_codes, event_minutes, receiver[pos], minutes[pos], BIG),
    "laundering": val["is_laundering"].to_numpy(),
})

quantiles = [0.5, 0.75, 0.9, 0.99]
summary = history.groupby("laundering")[["sender", "receiver"]].quantile(quantiles).unstack(0)
print("share of validation rows with at least 32 earlier transactions:")
print((history[["sender", "receiver"]] >= 32).mean().round(3).to_dict())
summary

share of validation rows with at least 32 earlier transactions:
{'sender': 0.819, 'receiver': 0.541}


sender            receiver       
laundering         0          1        0      1
0.50            59.0      29.00     33.0   24.0
0.75            94.0      67.00     50.0   44.0
0.90          6169.3   10211.00     72.0   80.0
0.99        129610.0  129535.15    127.0  142.3

Most accounts have short histories: the median validation row has 59 earlier transactions on the sender side and 33 on the receiver side, and laundering rows have fewer (29 and 24). About 10% of senders are hub accounts with thousands to 130,000 earlier transactions in at most eight days, so no fixed sequence length can hold their full history. A length of 64 is used for both sides: it covers the full history for about three quarters of laundering rows on each side, at a modest cost for a small model, while longer windows would add little before reaching the hub accounts. For hub senders the model therefore sees only their most recent 64 transactions, a span of minutes; if hub sent laundering stays undetected, this limit is the likely reason.

In [2]:
import time

from bankguard.data import load_accounts
from bankguard.features import build_features, fit_currency_median
from bankguard.sequence import build_events, gather, history_bounds

LENGTH = 64

accounts = load_accounts(DATA / "HI-Small_accounts.csv")
train = split_by_date(tx)["train"]
X = build_features(tx, accounts, fit_currency_median(train))

# Same 202,530 training rows as the text models.
y_train = train["is_laundering"].to_numpy()
rng = np.random.default_rng(42)
pos_rows = np.flatnonzero(y_train == 1)
neg_rows = rng.choice(np.flatnonzero(y_train == 0), size=200_000, replace=False)
sample = np.sort(np.concatenate([pos_rows, neg_rows]))
train_pos = train.index.to_numpy()[sample]
val_pos = val.index.to_numpy()
# Same validation fifth as the BERT runs, for early stopping.
eval_idx = np.random.default_rng(0).choice(len(val_pos), size=200_000, replace=False)

events = build_events(sender, receiver, minutes, self_tx)
tx_values = X[["log_amount", "amount_vs_currency", "self_transfer"]].to_numpy(np.float32)


def bounds(rows):
    """History positions of each row's sender and receiver."""
    return (
        history_bounds(events, sender[rows], minutes[rows], LENGTH),
        history_bounds(events, receiver[rows], minutes[rows], LENGTH),
    )


train_bounds, val_bounds = bounds(train_pos), bounds(val_pos)

print(f"events: {len(events.keys):,}")
print(f"training rows: {len(train_pos):,} ({y_train[sample].mean():.2%} laundering)")
print(f"missing values in event fields: {int(np.isnan(tx_values).sum())}")
for name, ((s0, s1), (r0, r1)) in [("train", train_bounds), ("validation", val_bounds)]:
    print(f"{name}: full 64-event history for {np.mean(s1 - s0 == LENGTH):.1%} of senders, "
          f"{np.mean(r1 - r0 == LENGTH):.1%} of receivers")

# Speed check: gather one batch of 256 rows, both sides.
rows = train_pos[:256]
(s0, s1), (r0, r1) = train_bounds
start = time.time()
for _ in range(20):
    gather(events, tx_values, s0[:256], s1[:256], minutes[rows], receiver[rows], LENGTH)
    gather(events, tx_values, r0[:256], r1[:256], minutes[rows], sender[rows], LENGTH)
ms = (time.time() - start) / 20 * 1000
print(f"gather: {ms:.1f} ms per batch, about {ms * len(train_pos) / 256 / 1000:.0f} s per epoch")

events: 9,565,478
training rows: 202,530 (1.25% laundering)
missing values in event fields: 0
train: full 64-event history for 11.3% of senders, 0.8% of receivers
validation: full 64-event history for 35.8% of senders, 8.3% of receivers
gather: 2.3 ms per batch, about 2 s per epoch


The event table holds 9,565,478 events (one per side of each transaction, self transfers once). The 202,530 training rows are the same as for the text models (1.25% laundering), and the same random fifth of validation is used for early stopping. Only two positions per side are stored for each row, and batches are gathered on the fly in about 2.3 ms, about 2 seconds per epoch, so data loading does not limit training.

Training rows have much shorter histories than validation rows: a full 64-event history exists for 11.3% of training senders and 0.8% of training receivers, against 35.8% and 8.3% in validation. The data starts on 1 Sep, so early training rows can only look back a day or two. Hub senders fill their 64 events within hours and are barely affected, but for ordinary accounts the model learns from shorter histories than it is scored on. The same 202,530 rows are kept so the comparison with the text models stays clean; this shift can only work against the sequence model. After the main run, the sequence model and LightGBM are both trained on the subset of these rows dated 3 to 6 Sep, to measure what the shift costs. That check is an ablation and does not enter model selection.

In [10]:
from bankguard.features import BREADTH, CATEGORICAL, FEATURES, GRAPH
from bankguard.model import fit_categories, gbm_frame

COUNTS = GRAPH + BREADTH + ["sender_entity_accounts", "receiver_entity_accounts"]
categories = fit_categories(X.loc[train.index], CATEGORICAL)


def encode(features, rows):
    """The 22 features as numbers: counts on a log scale, entity types one-hot."""
    frame = gbm_frame(features.iloc[rows], FEATURES, categories)
    numeric = frame.drop(columns=CATEGORICAL).astype("float32")
    numeric[COUNTS] = np.log1p(numeric[COUNTS])
    onehot = pd.get_dummies(frame[CATEGORICAL], dtype="float32")
    return pd.concat([numeric, onehot], axis=1)


train_rows, val_rows = encode(X, train_pos), encode(X, val_pos)
print("missing values:", int(train_rows.isna().sum().sum()), int(val_rows.isna().sum().sum()))
train_rows, val_rows = train_rows.fillna(0), val_rows.fillna(0)

# Standardise with training statistics only.
mean, std = train_rows.mean(), train_rows.std().replace(0, 1)
train_x = ((train_rows - mean) / std).to_numpy(np.float32)
val_x = ((val_rows - mean) / std).to_numpy(np.float32)

# The two amount fields of the events, standardised on training-period transactions.
amount_mean = tx_values[train.index, :2].mean(axis=0)
amount_std = tx_values[train.index, :2].std(axis=0)

print("row inputs:", train_x.shape[1], "columns:", list(train_rows.columns))
print("train", train_x.shape, "validation", val_x.shape)
del X

missing values: 0 0
row inputs: 32 columns: ['log_amount', 'amount_vs_currency', 'hour_sin', 'hour_cos', 'self_transfer', 'sender_out_1d', 'sender_in_1d', 'receiver_in_1d', 'receiver_out_1d', 'sender_out_3d', 'sender_in_3d', 'receiver_in_3d', 'receiver_out_3d', 'pair_prior_3d', 'sender_new_pair_out_3d', 'receiver_new_pair_in_3d', 'sender_entity_accounts', 'receiver_entity_accounts', 'same_entity', 'same_bank', 'sender_entity_type_Corporation', 'sender_entity_type_Country', 'sender_entity_type_Direct', 'sender_entity_type_Individual', 'sender_entity_type_Partnership', 'sender_entity_type_Sole Proprietorship', 'receiver_entity_type_Corporation', 'receiver_entity_type_Country', 'receiver_entity_type_Direct', 'receiver_entity_type_Individual', 'receiver_entity_type_Partnership', 'receiver_entity_type_Sole Proprietorship']
train (202530, 32) validation (965524, 32)


For the network, the 22 features become 32 numeric inputs: counts on a log scale, and the two entity types as one-hot columns over the six types seen in training (Corporation, Country, Direct, Individual, Partnership, Sole Proprietorship). There are no missing values. All inputs, and the two amount fields of the history events, are standardised with training statistics only.

In [4]:
import torch
from torch import nn

device = "cuda"


class HistoryEncoder(nn.Module):
    """A small transformer over one side's history, read out through a summary token."""

    def __init__(self, n_fields=6, width=64, heads=4, layers=2, dropout=0.1, length=LENGTH):
        super().__init__()
        self.embed = nn.Linear(n_fields, width)
        self.summary = nn.Parameter(torch.randn(1, 1, width) * 0.02)
        self.position = nn.Parameter(torch.randn(1, length + 1, width) * 0.02)
        layer = nn.TransformerEncoderLayer(
            width, heads, 4 * width, dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(width)

    def forward(self, values, mask):
        n = len(values)
        x = torch.cat([self.summary.expand(n, -1, -1), self.embed(values)], dim=1) + self.position
        keep = torch.cat([torch.ones(n, 1, dtype=torch.bool, device=mask.device), mask], dim=1)
        x = self.encoder(x, src_key_padding_mask=~keep)
        return self.norm(x[:, 0])


class SequenceModel(nn.Module):
    """The current row's features plus the sender's and the receiver's history."""

    def __init__(self, n_row, width=64, dropout=0.1):
        super().__init__()
        self.row = nn.Sequential(nn.Linear(n_row, width), nn.GELU())
        self.sender = HistoryEncoder(width=width, dropout=dropout)
        self.receiver = HistoryEncoder(width=width, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(3 * width, width), nn.GELU(), nn.Dropout(dropout), nn.Linear(width, 1)
        )

    def forward(self, row, sender_values, sender_mask, receiver_values, receiver_mask):
        parts = [
            self.row(row),
            self.sender(sender_values, sender_mask),
            self.receiver(receiver_values, receiver_mask),
        ]
        return self.head(torch.cat(parts, dim=1)).squeeze(1)


def make_batch(positions, row_x, row_bounds, idx):
    """Model inputs for rows `idx`: current-row features and both histories, on the GPU."""
    p = positions[idx]
    (s0, s1), (r0, r1) = row_bounds
    sides = []
    for start, end, counterparty in [(s0, s1, receiver[p]), (r0, r1, sender[p])]:
        values, mask = gather(events, tx_values, start[idx], end[idx], minutes[p], counterparty, LENGTH)
        values[..., :2] = (values[..., :2] - amount_mean) / amount_std * mask[..., None]
        sides += [torch.from_numpy(values).to(device), torch.from_numpy(mask).to(device)]
    return [torch.from_numpy(row_x[idx]).to(device), *sides]


model = SequenceModel(train_x.shape[1]).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
with torch.no_grad():
    logits = model(*make_batch(train_pos, train_x, train_bounds, np.arange(256)))
print(logits.shape, "any NaN:", bool(torch.isnan(logits).any()))

parameters: 224,065
torch.Size([256]) any NaN: False


The model has 224,065 parameters. Each side's history (64 events of 6 fields) is embedded to width 64, given learned positions and a summary token, and passed through two transformer layers with four attention heads, with padding masked out. The summary token's output summarises that side. The two summaries and an embedding of the current row's 32 inputs are joined and passed to a small classifier that outputs one score per transaction. A batch of 256 real rows runs without errors or missing values.

In [5]:
import copy

from sklearn.metrics import average_precision_score

BATCH = 256
y_val = val["is_laundering"].to_numpy()
y_fit = torch.tensor(y_train[sample], dtype=torch.float32, device=device)


@torch.no_grad()
def predict(model, positions, row_x, row_bounds, idx, batch=2048):
    """Laundering probability for rows `idx`."""
    model.eval()
    out = [torch.sigmoid(model(*make_batch(positions, row_x, row_bounds, idx[i:i + batch]))).cpu()
           for i in range(0, len(idx), batch)]
    return torch.cat(out).numpy()


def train_seed(seed, max_epochs=20, patience=3):
    """Train one model; keep the epoch with the best PR-AUC on the validation fifth."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = SequenceModel(train_x.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    best, best_state, wait, curve = -1.0, None, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        start, total = time.time(), 0.0
        order = rng.permutation(len(train_pos))
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]
            loss = loss_fn(model(*make_batch(train_pos, train_x, train_bounds, idx)), y_fit[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        pr_auc = average_precision_score(
            y_val[eval_idx], predict(model, val_pos, val_x, val_bounds, eval_idx))
        curve.append({"epoch": epoch, "train_loss": total / len(order), "eval_pr_auc": pr_auc,
                      "seconds": time.time() - start})
        print({k: round(v, 4) for k, v in curve[-1].items()})
        if pr_auc > best:
            best, best_state, wait = pr_auc, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, curve


# Probe: 50 training batches and 20,000 scored rows, for speed and memory.
probe = SequenceModel(train_x.shape[1]).to(device)
optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
probe_rows = np.random.default_rng(0).choice(len(train_pos), size=(50, BATCH), replace=False)
torch.cuda.reset_peak_memory_stats()
start = time.time()
for idx in probe_rows:
    loss = loss_fn(probe(*make_batch(train_pos, train_x, train_bounds, idx)), y_fit[idx])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
train_batch_s = (time.time() - start) / 50
start = time.time()
predict(probe, val_pos, val_x, val_bounds, eval_idx[:20_000])
score_row_s = (time.time() - start) / 20_000
epoch_s = train_batch_s * len(train_pos) / BATCH + score_row_s * len(eval_idx)
print(f"training: {train_batch_s * 1000:.0f} ms per batch; one epoch with evaluation: about {epoch_s:.0f} s")
print(f"scoring all validation: about {score_row_s * len(val_pos) / 60:.1f} min")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
del probe, optimizer

training: 71 ms per batch; one epoch with evaluation: about 72 s
scoring all validation: about 1.2 min
peak GPU memory: 0.51 GB


A probe of 50 training batches gives 71 ms per batch, about 72 seconds per epoch including evaluation on the validation fifth, and about 1.2 minutes to score all of validation, with 0.5 GB of peak GPU memory. Three seeds of at most 20 epochs therefore fit comfortably on the laptop.

In [6]:
import json

import mlflow

from bankguard.evaluate import evaluate

SEEDS = [42, 7, 123]
out_dir = Path("../models/sequence")
out_dir.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("bankguard-laundering")
results = []
with mlflow.start_run(run_name="Sequence model, history 64"):
    mlflow.log_params({
        "history_length": LENGTH, "width": 64, "heads": 4, "layers": 2, "dropout": 0.1,
        "learning_rate": 1e-3, "batch_size": BATCH, "max_epochs": 20, "patience": 3,
        "train_rows": len(train_pos), "row_inputs": train_x.shape[1], "event_fields": 6,
        "seeds": str(SEEDS),
    })
    for seed in SEEDS:
        with mlflow.start_run(run_name=f"Sequence model, seed {seed}", nested=True):
            model, curve = train_seed(seed)
            scores = predict(model, val_pos, val_x, val_bounds, np.arange(len(val_pos)))
            np.save(out_dir / f"sequence_val_scores_seed{seed}.npy", scores)
            torch.save(model.state_dict(), out_dir / f"sequence_seed{seed}.pt")

            metrics = evaluate(y_val, scores, f"Sequence model, seed {seed}")
            best_epoch = max(curve, key=lambda row: row["eval_pr_auc"])["epoch"]
            mlflow.log_param("seed", seed)
            mlflow.log_metrics({k.replace("%", "pct"): v for k, v in metrics.items() if k != "model"})
            mlflow.log_metrics({"best_epoch": best_epoch, "epochs_run": len(curve)})
            for row in curve:
                mlflow.log_metrics(
                    {"train_loss": row["train_loss"], "eval_pr_auc": row["eval_pr_auc"]},
                    step=row["epoch"],
                )
            results.append({**metrics, "seed": seed, "best_epoch": best_epoch, "epochs_run": len(curve)})

    table = pd.DataFrame(results)
    for key in ["pr_auc", "recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]:
        name = key.replace("%", "pct")
        mlflow.log_metric(f"{name}_mean", table[key].mean())
        mlflow.log_metric(f"{name}_std", table[key].std(ddof=0))

with open(out_dir / "sequence_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
table.round(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'epoch': 1, 'train_loss': 0.045, 'eval_pr_auc': 0.1688, 'seconds': 52.2463}
{'epoch': 2, 'train_loss': 0.0346, 'eval_pr_auc': 0.2559, 'seconds': 55.7827}
{'epoch': 3, 'train_loss': 0.0327, 'eval_pr_auc': 0.332, 'seconds': 54.8931}
{'epoch': 4, 'train_loss': 0.0312, 'eval_pr_auc': 0.3661, 'seconds': 61.1941}
{'epoch': 5, 'train_loss': 0.0296, 'eval_pr_auc': 0.3624, 'seconds': 50.5976}
{'epoch': 6, 'train_loss': 0.029, 'eval_pr_auc': 0.361, 'seconds': 50.1745}
{'epoch': 7, 'train_loss': 0.0283, 'eval_pr_auc': 0.3992, 'seconds': 50.9878}
{'epoch': 8, 'train_loss': 0.0276, 'eval_pr_auc': 0.4281, 'seconds': 51.2324}
{'epoch': 9, 'train_loss': 0.0271, 'eval_pr_auc': 0.4299, 'seconds': 50.9116}
{'epoch': 10, 'train_loss': 0.0264, 'eval_pr_auc': 0.388, 'seconds': 50.9548}
{'epoch': 11, 'train_loss': 0.0261, 'eval_pr_auc': 0.3963, 'seconds': 51.6772}
{'epoch': 12, 'train_loss': 0.0258, 'eval_pr_auc': 0.4229, 'seconds': 51.4797}
{'epoch': 1, 'train_loss': 0.0473, 'eval_pr_auc': 0.1804, 'seconds

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,seed,best_epoch,epochs_run
0,"Sequence model, seed 42",0.4544,0.5241,0.8292,42,9,12
1,"Sequence model, seed 7",0.4820,0.5396,0.8349,7,17,20
2,"Sequence model, seed 123",0.4338,0.4913,0.8137,123,5,8


Each seed trained until its PR-AUC on the validation fifth had not improved for three epochs. The best epochs were 9, 17 and 5; seed 7 reached the 20 epoch cap while still improving slowly, and the validation curves fluctuated by several points between epochs, as is common for a model trained from scratch. On all of validation, single seeds reach PR-AUC 0.434 to 0.482 (mean 0.457, standard deviation 0.020), recall 0.491 to 0.540 at a 0.1% false positive rate and 0.814 to 0.835 at 1%. Every seed is ahead of TabPFN-3.5-Fast (0.398, 0.397, 0.796) and LightGBM (0.341, 0.379, 0.764) on all three metrics. The spread across seeds is larger than LightGBM's (0.331 to 0.342), so single runs of this model are less stable.

In [7]:
from bankguard.data import load_patterns, typology_for
from bankguard.evaluate import laundering_group, recall_by_group

seed_scores = np.stack([np.load(out_dir / f"sequence_val_scores_seed{s}.npy") for s in SEEDS])
avg_scores = seed_scores.mean(axis=0)
np.save(out_dir / "sequence_val_scores_avg.npy", avg_scores)

models = {
    "Sequence, average of 3 seeds": avg_scores,
    "TabPFN-3.5-Fast": np.load("../models/tabpfn/tabpfn_val_scores.npy"),
    "LightGBM": np.load("../models/lightgbm_val_scores.npy"),
}
avg_metrics = evaluate(y_val, avg_scores, "Sequence model, average of 3 seeds")
metric_cols = ["pr_auc", "recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]
overall = pd.DataFrame([
    avg_metrics,
    {"model": "Sequence model, single seed mean", **table[metric_cols].mean()},
    {"model": "Sequence model, single seed std", **table[metric_cols].std(ddof=0)},
    *[evaluate(y_val, s, name) for name, s in list(models.items())[1:]],
])
display(overall.round(4))

# Log the averaged entry to the parent MLflow run.
parent = mlflow.search_runs(
    experiment_names=["bankguard-laundering"],
    filter_string="tags.mlflow.runName = 'Sequence model, history 64'",
).loc[0, "run_id"]
with mlflow.start_run(run_id=parent):
    mlflow.log_metrics({f"avg_{k.replace('%', 'pct')}": v for k, v in avg_metrics.items() if k != "model"})

# Recall by laundering group, at a 1% false positive rate.
patterns = load_patterns(DATA / "HI-Small_Patterns.txt")
sender_out_3d = np.rint(np.expm1(val_rows["sender_out_3d"].to_numpy()))
val_group = laundering_group(typology_for(val, patterns), sender_out_3d)
by_group = pd.concat(
    {name: recall_by_group(y_val, s, val_group)["recall_at_1.0%_fpr"] for name, s in models.items()},
    axis=1,
)
by_group.insert(0, "laundering", recall_by_group(y_val, avg_scores, val_group)["laundering"])
display(by_group.round(3))

# Recall by length of the sender's history, at a 1% false positive rate.
(s0, s1), _ = val_bounds
bucket = pd.cut(s1 - s0, [-1, 0, 15, 63, 64], labels=["none", "1 to 15", "16 to 63", "full 64"])
laundering = y_val == 1
by_length = pd.DataFrame({"laundering": pd.Series(bucket[laundering]).value_counts().sort_index()})
for name, s in models.items():
    caught = s > np.quantile(s[y_val == 0], 0.99)
    by_length[name] = [caught[laundering & (bucket == b)].mean() for b in bucket.categories]
by_length.round(3)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,"Sequence model, average of 3 seeds",0.4855,0.5270,0.8388
1,"Sequence model, single seed mean",0.4568,0.5183,0.8259
2,"Sequence model, single seed std",0.0198,0.0201,0.0090
3,TabPFN-3.5-Fast,0.3985,0.3967,0.7963
4,LightGBM,0.3412,0.3793,0.7635


,laundering,"Sequence, average of 3 seeds",TabPFN-3.5-Fast,LightGBM
patterned,657,0.989,0.945,0.913
"unpatterned, quiet sender",246,0.890,0.829,0.764
"unpatterned, hub sender",133,0.000,0.000,0.023


,laundering,"Sequence, average of 3 seeds",TabPFN-3.5-Fast,LightGBM
none,13,0.923,0.923,0.923
1 to 15,380,0.937,0.879,0.821
16 to 63,418,0.978,0.931,0.902
full 64,225,0.409,0.400,0.400


Averaging the three seeds gives PR-AUC 0.486, recall 0.527 at a 0.1% false positive rate and 0.839 at 1%, the best of all models so far. Averaging adds about 0.03 PR-AUC over a single seed, as expected from an ensemble; this entry is used in the comparison, as decided before the results, and is labelled as an average.

By group, at a 1% false positive rate, the sequence model catches 98.9% of patterned laundering (TabPFN 94.5%, LightGBM 91.3%) and 89.0% of unpatterned laundering from quiet senders (82.9% and 76.4%). Laundering from hub senders is missed entirely, 0 of 133, as by every other model. Overall recall of 83.9% is close to the 87.2% that is possible while hub senders go undetected, so the model finds about 96% of the laundering that the inputs can separate.

By length of the sender's history, the gain comes from accounts with 1 to 63 earlier transactions (93.7% and 97.8% recall, against 82.1% to 93.1% for the other models). Rows with a full 64-event history stay at about 40% for all three models, because 133 of their 225 laundering rows come from hub senders. For a hub, 64 events cover only a few minutes of activity, and the recent history carries no signal that separates its laundering. The hypothesis that recent history would detect hub sent laundering is therefore not supported; detecting it likely needs structure across many accounts, such as a graph model.

Two cautions. The "same counterparty" field is related to the pair count, which is partly a signal of the simulator, so part of the gain may not transfer to real data. And earlier transactions of the same laundering scheme appear in the history, which is legitimate (no labels are used and only earlier transactions are seen) but makes structured typologies easier to recognise than they may be in practice.

## Time shift check

Training rows have shorter histories than validation rows because the data starts on 1 Sep. As fixed in the PRD before the main run, the sequence model and LightGBM are trained again on the subset of the 202,530 training rows dated 3 to 6 Sep, with otherwise unchanged settings, and scored on all of validation. LightGBM is also trained on all 202,530 rows, so both models have a matching pair. This check is an ablation and does not enter model selection.

In [8]:
import mlflow

from bankguard.evaluate import evaluate
from bankguard.model import fit_gbm
from bankguard.model import predict as gbm_predict

SEEDS = [42, 7, 123]
out_dir = Path("../models/sequence")
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("bankguard-laundering")

late = tx["date"].to_numpy()[train_pos] >= np.datetime64("2022-09-03")
y_sample = y_train[sample]
print(f"rows dated 3 to 6 Sep: {late.sum():,} of {len(late):,}; "
      f"laundering {int(y_sample[late].sum())} of {int(y_sample.sum())}")


def subset(row_bounds, mask):
    """History positions for the rows in `mask` only."""
    return tuple((start[mask], end[mask]) for start, end in row_bounds)


# Sequence model on the later rows. train_seed reads these four names, so they are swapped
# for the subset during the loop and always restored afterwards.
full = (train_pos, train_x, train_bounds, y_fit)
train_pos, train_x, train_bounds, y_fit = (
    full[0][late], full[1][late], subset(full[2], late), full[3][torch.from_numpy(late).to(device)]
)
try:
    late_scores = []
    for seed in SEEDS:
        model, curve = train_seed(seed)
        late_scores.append(predict(model, val_pos, val_x, val_bounds, np.arange(len(val_pos))))
finally:
    train_pos, train_x, train_bounds, y_fit = full
late_avg = np.mean(late_scores, axis=0)
np.save(out_dir / "sequence_val_scores_late_avg.npy", late_avg)

# LightGBM, with the same function and settings as in 02_baselines, on both sets of rows.
X = build_features(tx, accounts, fit_currency_median(train))
X_sample, X_val = X.iloc[train_pos], X.iloc[val_pos]
gbm_scores = {}
for name, mask in [("all", np.ones(len(train_pos), bool)), ("late", late)]:
    gbm = fit_gbm(X_sample[mask], y_sample[mask], X_val, y_val, FEATURES, categories)
    gbm_scores[name] = gbm_predict(gbm, X_val, FEATURES, categories)
del X, X_sample

check = pd.DataFrame([
    evaluate(y_val, np.load(out_dir / "sequence_val_scores_avg.npy"), "Sequence, all 202,530 rows (main run)"),
    evaluate(y_val, late_avg, "Sequence, 3 to 6 Sep rows"),
    evaluate(y_val, gbm_scores["all"], "LightGBM, all 202,530 rows"),
    evaluate(y_val, gbm_scores["late"], "LightGBM, 3 to 6 Sep rows"),
])
late_single = [evaluate(y_val, s, "")["pr_auc"] for s in late_scores]
print("sequence, 3 to 6 Sep, single seeds:", [round(v, 4) for v in late_single])

with mlflow.start_run(run_name="Time shift check (ablation)"):
    mlflow.set_tag("ablation", "time shift")
    mlflow.log_param("late_rows", int(late.sum()))
    for key, row in zip(["seq_all", "seq_late", "gbm_all", "gbm_late"], check.to_dict("records")):
        mlflow.log_metrics({f"{key}_{k.replace('%', 'pct')}": v for k, v in row.items() if k != "model"})
check.round(4)

rows dated 3 to 6 Sep: 86,816 of 202,530; laundering 1800 of 2530
{'epoch': 1, 'train_loss': 0.0584, 'eval_pr_auc': 0.2797, 'seconds': 31.9434}
{'epoch': 2, 'train_loss': 0.0391, 'eval_pr_auc': 0.3582, 'seconds': 28.9785}
{'epoch': 3, 'train_loss': 0.0365, 'eval_pr_auc': 0.2821, 'seconds': 29.2562}
{'epoch': 4, 'train_loss': 0.0334, 'eval_pr_auc': 0.3924, 'seconds': 28.2234}
{'epoch': 5, 'train_loss': 0.032, 'eval_pr_auc': 0.4272, 'seconds': 28.2974}
{'epoch': 6, 'train_loss': 0.0305, 'eval_pr_auc': 0.4354, 'seconds': 28.4401}
{'epoch': 7, 'train_loss': 0.0293, 'eval_pr_auc': 0.4572, 'seconds': 28.7989}
{'epoch': 8, 'train_loss': 0.0298, 'eval_pr_auc': 0.4598, 'seconds': 28.8574}
{'epoch': 9, 'train_loss': 0.0286, 'eval_pr_auc': 0.4755, 'seconds': 28.8572}
{'epoch': 10, 'train_loss': 0.0283, 'eval_pr_auc': 0.4729, 'seconds': 28.8747}
{'epoch': 11, 'train_loss': 0.0279, 'eval_pr_auc': 0.4805, 'seconds': 29.0986}
{'epoch': 12, 'train_loss': 0.0284, 'eval_pr_auc': 0.4843, 'seconds': 29.08

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,"Sequence, all 202,530 rows (main run)",0.4855,0.5270,0.8388
1,"Sequence, 3 to 6 Sep rows",0.5141,0.5531,0.8369
2,"LightGBM, all 202,530 rows",0.2859,0.3311,0.7625
3,"LightGBM, 3 to 6 Sep rows",0.3482,0.3919,0.7876


The rows dated 3 to 6 Sep are 86,816 of the 202,530 training rows but hold 1,800 of the 2,530 laundering rows, so this subset changes two things at once: longer histories and a higher laundering share (2.1% instead of 1.25%).

History still helps when both models train on later rows: on the 3 to 6 Sep rows the sequence model reaches PR-AUC 0.514 (single seeds 0.513, 0.495 and 0.486) against 0.348 for LightGBM, a gap of 0.17, close to the 0.20 gap on all 202,530 rows (0.486 against 0.286). The sequence model's lead is therefore not an artifact of the shorter training histories.

Both models improve when the rows from 1 and 2 Sep are dropped, despite losing 730 laundering rows: the sequence model by 0.029 and LightGBM by 0.062. Rows from the first days, whose histories and three-day counts are cut short by the start of the data, appear to mislead training, LightGBM's trailing counts more than the sequence model's 64-event histories; part of the gain may also come from the higher laundering share, which this check cannot separate. LightGBM trained on the same 202,530 rows scores 0.286, against 0.341 when trained on all 3.25M training rows, so more normal rows help it; the sequence model was trained only on the 202,530 rows, and whether more rows would help it too is untested.

As fixed before the run, this check is an ablation and does not change model selection, which uses the main run. For a future version, the first days would be used only as history and not as training rows, decided in advance.

## Test set predictions

The chosen model is scored on the test rows (9 to 10 Sep) with the three saved seed weights, averaged as on validation. Only the predictions are saved here; they are evaluated against the test labels in the comparison notebook, together with the other models.

In [11]:
SEEDS = [42, 7, 123]
out_dir = Path("../models/sequence")

test_pos = split_by_date(tx)["test"].index.to_numpy()
X = build_features(tx, accounts, fit_currency_median(train))
test_x = ((encode(X, test_pos).fillna(0) - mean) / std).to_numpy(np.float32)
del X
test_bounds = bounds(test_pos)

test_scores = []
for seed in SEEDS:
    model = SequenceModel(train_x.shape[1]).to(device)
    model.load_state_dict(torch.load(out_dir / f"sequence_seed{seed}.pt", map_location=device))
    # Check: the reloaded model reproduces its saved validation scores.
    again = predict(model, val_pos, val_x, val_bounds, np.arange(5_000))
    saved = np.load(out_dir / f"sequence_val_scores_seed{seed}.npy")[:5_000]
    print(f"seed {seed}: largest difference from saved validation scores {np.abs(again - saved).max():.2e}")
    test_scores.append(predict(model, test_pos, test_x, test_bounds, np.arange(len(test_pos))))

test_avg = np.mean(test_scores, axis=0)
np.save(out_dir / "sequence_test_scores_avg.npy", test_avg)
print(f"test rows: {len(test_pos):,}; scores: {test_avg.shape[0]:,}; missing: {int(np.isnan(test_avg).sum())}")

seed 42: largest difference from saved validation scores 0.00e+00
seed 7: largest difference from saved validation scores 0.00e+00
seed 123: largest difference from saved validation scores 0.00e+00
test rows: 862,792; scores: 862,792; missing: 0


Each reloaded seed reproduces its saved validation scores exactly (largest difference 0), confirming that the weights and preprocessing match training. The 862,792 test rows are scored with each seed and averaged; the predictions are saved without reading the test labels, which are used only in the comparison notebook.